In [10]:
from pathlib import Path

UCF_ROOT = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/"
HMDB_ROOT = "/kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/"

# ALT_PKG: update this to wherever your alt package folder lives
ALT_PKG = "/kaggle/input/datasets/darsan251503/alt-optionb-code/alt  copy 2"  # ← set this to your actual alt package path

print("UCF_ROOT :", UCF_ROOT)
print("HMDB_ROOT:", HMDB_ROOT)
print("ALT_PKG  :", ALT_PKG)

UCF_ROOT : /kaggle/input/datasets/matthewjansen/ucf101-action-recognition/
HMDB_ROOT: /kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/
ALT_PKG  : /kaggle/input/datasets/darsan251503/alt-optionb-code/alt  copy 2


In [11]:
import shutil, os

TARGET = "/kaggle/working/alt"

# Remove previous copied project only
if os.path.exists(TARGET):
    shutil.rmtree(TARGET)

# Copy dataset folder
shutil.copytree(ALT_PKG, TARGET)

# Enter project
os.chdir(TARGET)

print("Working dir:", os.getcwd())
print("Contents:", os.listdir(".")[:10])

Working dir: /kaggle/working/alt
Contents: ['.env', 'eval.py', 'train.py', 'corpus', 'configs', 'training', 'data', 'utils', 'models']


In [12]:
import os
import json
import shutil
from pathlib import Path

HMDB_SPLITS_SRC = "/kaggle/input/datasets/darsan251503/hmdb-traintestsplit/test_train_splits"
HMDB_SPLITS_DST = "/tmp/hmdb_splits/testTrainMultipleSplits"
OUT_DIR = "/kaggle/working/alt/corpus"
LABEL_OFFSET = 101

os.makedirs(HMDB_SPLITS_DST, exist_ok=True)
for fname in os.listdir(HMDB_SPLITS_SRC):
    if fname.endswith(".txt"):
        shutil.copy(os.path.join(HMDB_SPLITS_SRC, fname), HMDB_SPLITS_DST)
print("HMDB split files copied:", len(os.listdir(HMDB_SPLITS_DST)))

hmdb_classes = sorted({
    p.stem.replace("_test_split1", "")
    for p in Path(HMDB_SPLITS_DST).glob("*_test_split1.txt")
})
hmdb_class_map = {name: i for i, name in enumerate(hmdb_classes)}

def build_hmdb(split_num, out_path, flag):
    entries = []
    for p in sorted(Path(HMDB_SPLITS_DST).glob(f"*_test_split{split_num}.txt")):
        class_name = p.stem.replace(f"_test_split{split_num}", "")
        label = hmdb_class_map[class_name] + LABEL_OFFSET
        with open(p) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 2:
                    continue
                fname, file_flag = parts
                if int(file_flag) == flag:
                    entries.append({
                        "path": f"{class_name}/{fname}",
                        "label": label,
                        "dataset": "hmdb",
                    })
    with open(out_path, "w") as handle:
        json.dump(entries, handle, indent=2)
    print(f"Saved {len(entries)} entries -> {out_path}")

build_hmdb(split_num=1, out_path=f"{OUT_DIR}/hmdb_val_split1.json", flag=2)
build_hmdb(split_num=2, out_path=f"{OUT_DIR}/hmdb_val_split2.json", flag=2)
build_hmdb(split_num=3, out_path=f"{OUT_DIR}/hmdb_val_split3.json", flag=2)
print("HMDB split JSONs generated.")


HMDB split files copied: 153
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split1.json
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split2.json
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split3.json
HMDB split JSONs generated.


In [13]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git",
    "git+https://github.com/facebookresearch/ToMe.git",
    "decord",
], check=True)

import sys, os
from pathlib import Path

ALT_DST = "/kaggle/working/alt"
sys.path.insert(0, ALT_DST)
os.chdir(ALT_DST)
print("CWD:", os.getcwd())

# Quick import + upgraded-code verification
from models.alt import ALT
from training.trainer import Trainer
clip_encoder_path = Path("/kaggle/working/alt/models/clip_encoder.py")
clip_encoder_text = clip_encoder_path.read_text(encoding="utf-8")
print("Imports OK")
print("clip_encoder.py mtime:", clip_encoder_path.stat().st_mtime)
print("Has _merge_tokens:", "def _merge_tokens" in clip_encoder_text)


CWD: /kaggle/working/alt
Imports OK
clip_encoder.py mtime: 1778440696.3735263
Has _merge_tokens: True


In [14]:
import os
import yaml
from pprint import pprint

CFG_PATH = "/kaggle/working/alt/configs/alt_b16.yaml"
cfg = yaml.safe_load(open(CFG_PATH, "r"))

cfg["clip_model"] = "ViT-B/16"
cfg["corpus_path"] = "/kaggle/working/alt/corpus/corpus_embeddings.pt"
cfg["label_file"] = "/kaggle/working/alt/corpus/all_labels_ordered.json"
cfg["train_json"] = "/kaggle/working/alt/corpus/ucf_train_split1.json"
cfg["val_json"] = "/kaggle/working/alt/corpus/ucf_val_split1.json"
cfg["data_roots"] = {"ucf": UCF_ROOT, "hmdb": HMDB_ROOT}

cfg["corpus_build_if_missing"] = True
cfg["corpus_rebuild"] = True
cfg["corpus_use_t5_filter"] = True
cfg["corpus_use_lesk"] = True
cfg["corpus_t5_model"] = "t5-small"
cfg["early_stopping_patience"] = 5
cfg["epochs"] = 30

# Optional runtime overrides (keep defaults unless experimenting)
# cfg["tome_r"] = 4
# cfg["tome_layers"] = None
# cfg["tau"] = 1.0
# cfg["alignment_topk"] = None

with open(CFG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config patched:")
pprint(cfg)

print("\nPath checks:")
for k in ["corpus_path", "label_file", "train_json", "val_json"]:
    print(f"  {k}: {cfg[k]} -> exists={os.path.exists(cfg[k])}")


Config patched:
{'adapter_blocks': 4,
 'adapter_lr': 8e-05,
 'alignment_topk': None,
 'backbone_lr': 8e-06,
 'batch_size': 16,
 'clip_model': 'ViT-B/16',
 'corpus_build_if_missing': True,
 'corpus_path': '/kaggle/working/alt/corpus/corpus_embeddings.pt',
 'corpus_rebuild': True,
 'corpus_t5_model': 't5-small',
 'corpus_use_lesk': True,
 'corpus_use_t5_filter': True,
 'data_roots': {'hmdb': '/kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/',
                'ucf': '/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/'},
 'early_stopping_patience': 5,
 'epochs': 30,
 'frame_size': 224,
 'freeze_backbone': True,
 'grad_accum_steps': 2,
 'label_file': '/kaggle/working/alt/corpus/all_labels_ordered.json',
 'log_interval': 50,
 'n_heads': 8,
 'num_classes': 152,
 'num_frames': 8,
 'num_workers': 2,
 'seed': 42,
 'tau': 1.0,
 'tome_layers': None,
 'tome_r': 8,
 'train_json': '/kaggle/working/alt/corpus/ucf_train_split1.json',
 'use_temporal_pe': False,
 'val_json': '/

In [16]:
import os
import torch
from corpus.build_corpus import build_corpus

cfg = yaml.safe_load(open("/kaggle/working/alt/configs/alt_b16.yaml", "r"))
device = "cuda" if torch.cuda.is_available() else "cpu"

llm_provider = os.getenv("LLM_PROVIDER", "transformers")
llm_model = os.getenv("LLM_MODEL", "google/flan-t5-small")

if llm_provider == "groq" and not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY for Groq expansion.")
if llm_provider == "openai" and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY for OpenAI expansion.")

build_corpus(
    label_file=cfg["label_file"],
    output_dir="/kaggle/working/alt/corpus",
    output_path=cfg["corpus_path"],
    device=device,
    use_lesk=cfg.get("corpus_use_lesk", False),
    use_t5_filter=cfg.get("corpus_use_t5_filter", False),
    t5_model_name=cfg.get("corpus_t5_model", "t5-small"),
    expand_corpus=True,
    llm_provider=llm_provider,
    llm_model_name=llm_model,
    expansions_per_noun=4,
    min_corpus_size=500,
    body_parts_file="/kaggle/working/alt/corpus/body_parts.json",
    expansion_cache="/kaggle/working/alt/corpus/llm_expansions.json",
)
print("Expanded corpus ready:", cfg["corpus_path"])


Ensuring NLTK resource: wordnet
Ensuring NLTK resource: omw-1.4


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
100%|████████████████████████████████████████| 335M/335M [00:02<00:00, 129MiB/s]


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Corpus saved: torch.Size([548, 512]) → /kaggle/working/alt/corpus/corpus_embeddings.pt
Expanded corpus ready: /kaggle/working/alt/corpus/corpus_embeddings.pt


In [18]:
import json
import subprocess
from collections import defaultdict
from pathlib import Path

import torch
from models.alt import ALT

# 1) Config sanity checks
cfg = yaml.safe_load(open("/kaggle/working/alt/configs/alt_b16.yaml", "r"))
assert cfg["train_json"].endswith("ucf_train_split1.json"), "train_json should be UCF-only"
assert cfg["val_json"].endswith("ucf_val_split1.json"), "val_json should be UCF-only"
assert cfg.get("corpus_use_lesk") is True, "corpus_use_lesk must be true"
assert int(cfg.get("early_stopping_patience", 0)) == 5, "early_stopping_patience must be 5"
print("Config sanity: OK")

# 2) Corpus build smoke (LLM expansion + Lesk)
build_corpus(
    label_file=cfg["label_file"],
    output_dir="/kaggle/working/alt/corpus",
    output_path=cfg["corpus_path"],
    device=device,
    use_lesk=cfg.get("corpus_use_lesk", False),
    use_t5_filter=cfg.get("corpus_use_t5_filter", False),
    t5_model_name=cfg.get("corpus_t5_model", "t5-small"),
    expand_corpus=True,
    llm_provider=llm_provider,
    llm_model_name=llm_model,
    expansions_per_noun=4,
    min_corpus_size=500,
    body_parts_file="/kaggle/working/alt/corpus/body_parts.json",
    expansion_cache="/kaggle/working/alt/corpus/llm_expansions.json",
)
print("Corpus build smoke: OK")

# 3) Tiny HMDB eval sanity
hmdb_val_path = "/kaggle/working/alt/corpus/hmdb_val_split1.json"
val_entries = json.load(open(hmdb_val_path, "r"))
bucket = defaultdict(list)
for e in val_entries:
    bucket[e["label"]].append(e)
selected = []
for label in list(bucket.keys())[:2]:
    selected.extend(bucket[label][:5])
tiny_split = "/kaggle/working/alt/corpus/hmdb_val_tiny.json"
with open(tiny_split, "w") as f:
    json.dump(selected, f)
print("Sanity split size:", len(selected))
if len(selected) == 0:
    raise RuntimeError("No HMDB samples found for sanity split.")

S = torch.load(cfg["corpus_path"], map_location="cpu").float()
tmp_model = ALT(cfg, S)
tmp_ckpt = "/kaggle/working/runs/alt_b16/init_sanity.pt"
Path("/kaggle/working/runs/alt_b16").mkdir(parents=True, exist_ok=True)
torch.save({"epoch": 0, "best_acc": 0.0, "model_state": tmp_model.state_dict()}, tmp_ckpt)

subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config", "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint", tmp_ckpt,
    "--split_json", tiny_split,
    "--label_offset", "101",
    "--n_classes", "51",
] , check=True)
print("Tiny zero-shot sanity check completed.")


Config sanity: OK
Ensuring NLTK resource: wordnet
Ensuring NLTK resource: omw-1.4


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Corpus saved: torch.Size([548, 512]) → /kaggle/working/alt/corpus/corpus_embeddings.pt
Corpus build smoke: OK
Sanity split size: 10
Loaded checkpoint (epoch 0, best_acc=0.0000)
hmdb_val_tiny.json: 0.0000 (0/10)

Zero-shot accuracy on split: 0.0000
Tiny zero-shot sanity check completed.


In [19]:
import yaml
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms

cfg = yaml.safe_load(open("/kaggle/working/alt/configs/alt_b16.yaml", "r"))

train_ds = VideoDataset(
    cfg["train_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_train_transforms(cfg["frame_size"]),
    mode="train",
)
val_ds = VideoDataset(
    cfg["val_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_val_transforms(cfg["frame_size"]),
    mode="val",
)

# Verification: dataset creation succeeds without monkey-patch
print("VideoDataset uses built-in resolver:", hasattr(train_ds, "_resolve_path"))
print("train_ds size:", len(train_ds), "val_ds size:", len(val_ds))
print("Resolved sample path:", train_ds._resolve_path(train_ds.entries[0]))

VideoDataset uses built-in resolver: True
train_ds size: 9537 val_ds size: 3783
Resolved sample path: /kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/ApplyEyeMakeup/v_ApplyEyeMakeup_g08_c01.avi


In [20]:
import json
import subprocess
from collections import defaultdict
from pathlib import Path

# Load HMDB val split (not the combined val_json)
hmdb_val_path = "/kaggle/working/alt/corpus/hmdb_val_split1.json"
val_entries   = json.load(open(hmdb_val_path, "r"))

# Group by label, take first 2 classes, up to 5 videos each
bucket = defaultdict(list)
for e in val_entries:
    bucket[e["label"]].append(e)

selected = []
for label in list(bucket.keys())[:2]:
    selected.extend(bucket[label][:5])

tiny_split = "/kaggle/working/alt/corpus/hmdb_val_tiny.json"
with open(tiny_split, "w") as f:
    json.dump(selected, f)

# Build a dummy checkpoint for the sanity run
import torch
from models.alt import ALT
S = torch.load(cfg["corpus_path"], map_location="cpu").float()
tmp_model = ALT(cfg, S)   # uses cfg, includes ToMe, alignment, adapter
tmp_ckpt = "/kaggle/working/runs/alt_b16/init_sanity.pt"
Path("/kaggle/working/runs/alt_b16").mkdir(parents=True, exist_ok=True)
torch.save({"epoch": 0, "best_acc": 0.0, "model_state": tmp_model.state_dict()}, tmp_ckpt)

print("Sanity split size:", len(selected))
if len(selected) == 0:
    raise RuntimeError("No HMDB samples found for sanity split.")

# Run eval on the tiny split
subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config",       "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint",   tmp_ckpt,
    "--split_json",   tiny_split,
    "--label_offset", "101",
    "--n_classes",    "51",
], check=True)
print("Tiny zero-shot sanity check completed.")

Sanity split size: 10
Loaded checkpoint (epoch 0, best_acc=0.0000)
hmdb_val_tiny.json: 0.0000 (0/10)

Zero-shot accuracy on split: 0.0000
Tiny zero-shot sanity check completed.


In [21]:
import os
required = [
    '/kaggle/working/alt/corpus/corpus_embeddings.pt',
    '/kaggle/working/alt/corpus/all_labels_ordered.json',
    '/kaggle/working/alt/corpus/ucf_train_split1.json',
    '/kaggle/working/alt/corpus/ucf_val_split1.json',
    '/kaggle/working/alt/corpus/hmdb_val_split1.json',
    '/kaggle/working/alt/models/alt.py',
    '/kaggle/working/alt/models/clip_encoder.py',
    '/kaggle/working/alt/models/alignment.py',
    '/kaggle/working/alt/models/video_adapter.py',
    '/kaggle/working/alt/data/video_dataset.py',
    '/kaggle/working/alt/data/transforms.py',
    '/kaggle/working/alt/training/trainer.py',
    '/kaggle/working/alt/training/loss.py',
    '/kaggle/working/alt/train.py',
    '/kaggle/working/alt/eval.py',
    '/kaggle/working/alt/configs/alt_b16.yaml',
]
for f in required:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")


✓  /kaggle/working/alt/corpus/corpus_embeddings.pt
✓  /kaggle/working/alt/corpus/all_labels_ordered.json
✓  /kaggle/working/alt/corpus/ucf_train_split1.json
✓  /kaggle/working/alt/corpus/ucf_val_split1.json
✓  /kaggle/working/alt/corpus/hmdb_val_split1.json
✓  /kaggle/working/alt/models/alt.py
✓  /kaggle/working/alt/models/clip_encoder.py
✓  /kaggle/working/alt/models/alignment.py
✓  /kaggle/working/alt/models/video_adapter.py
✓  /kaggle/working/alt/data/video_dataset.py
✓  /kaggle/working/alt/data/transforms.py
✓  /kaggle/working/alt/training/trainer.py
✓  /kaggle/working/alt/training/loss.py
✓  /kaggle/working/alt/train.py
✓  /kaggle/working/alt/eval.py
✓  /kaggle/working/alt/configs/alt_b16.yaml


In [22]:
import yaml, torch
from models.alt import ALT

cfg = yaml.safe_load(open('/kaggle/working/alt/configs/alt_b16.yaml', 'r'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

S = torch.load(cfg['corpus_path'], map_location='cpu').float()
model = ALT(cfg, S).to(device).eval()

with torch.no_grad():
    dummy = torch.randn(2, cfg['num_frames'], 3, cfg['frame_size'], cfg['frame_size'], device=device)
    token_feats = model.encoder(dummy)

print('Encoder token shape:', tuple(token_feats.shape))
token_count = token_feats.shape[2]
if cfg.get('tome_r', 0) > 0 and token_count >= 197:
    raise RuntimeError(f'ToMe seems inactive: got {token_count} tokens (expected < 197).')
print('ToMe verification passed.')


Encoder token shape: (2, 8, 101, 512)
ToMe verification passed.


In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/alt")
os.chdir("/kaggle/working/alt")

import yaml, torch
from models.alt import ALT
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms
from training.trainer import Trainer
from training.loss import build_label_embeddings

cfg = yaml.safe_load(open("configs/alt_b16.yaml", "r"))
# Recommended smoke run before full training:
# cfg["epochs"] = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S = torch.load(cfg["corpus_path"], map_location="cpu").float()
model = ALT(cfg, S)
train_ds = VideoDataset(cfg["train_json"], cfg["data_roots"], cfg["num_frames"], get_train_transforms(cfg["frame_size"]), "train")
val_ds = VideoDataset(cfg["val_json"], cfg["data_roots"], cfg["num_frames"], get_val_transforms(cfg["frame_size"]), "val")
C = build_label_embeddings(cfg["label_file"], device)
torch.cuda.empty_cache()
print("Cleared CUDA cache after label embeddings")

trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
trainer.run(start_epoch=0, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")


Cleared CUDA cache after label embeddings
  ep0 step0/596 loss=6.9269 lr=0.00e+00


/kaggle/working/alt/training/trainer.py:124: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()


  ep0 step50/596 loss=7.9390 lr=6.71e-07


In [ ]:
import glob
import torch

candidates = sorted(set(
    glob.glob("/kaggle/working/runs/alt_b16/latest.pt") +
    glob.glob("/kaggle/input/*/latest.pt") +
    glob.glob("/kaggle/input/**/*latest.pt", recursive=True)
))
print("Found checkpoints:")
for c in candidates:
    print(" -", c)

if candidates:
    ranked = []
    for path in candidates:
        try:
            ck = torch.load(path, map_location="cpu")
            ranked.append((int(ck.get("epoch", -1)), path))
        except Exception as e:
            print(f"Skipping unreadable checkpoint {path}: {e}")
    ranked.sort(key=lambda x: x[0])
    best_epoch, best_path = ranked[-1]
    print(f"Resuming from: {best_path} (epoch={best_epoch})")

    trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
    last_ep = trainer.load_checkpoint(best_path)
    print(f"Loaded checkpoint epoch={last_ep}")
    trainer.run(start_epoch=last_ep + 1, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")
else:
    print("No checkpoint found to resume.")


In [9]:
!kaggle kernels output darsan251503/altfinal-ucftrain-hmdb-val-corpusllm -p /kaggle/working/

Output file downloaded to /kaggle/working/alt/.env
alt_b16.yaml: Skipping, found more recently modified local copy (use --force to force download)
Output file downloaded to /kaggle/working/alt/configs/alt_b16_local.yaml
Output file downloaded to /kaggle/working/alt/corpus/__pycache__/build_corpus.cpython-312.pyc
Output file downloaded to /kaggle/working/alt/corpus/action_labels.json
Output file downloaded to /kaggle/working/alt/corpus/all_labels_ordered.json
Output file downloaded to /kaggle/working/alt/corpus/body_parts.json
Output file downloaded to /kaggle/working/alt/corpus/build_action_labels.py
Output file downloaded to /kaggle/working/alt/corpus/build_corpus.py
Output file downloaded to /kaggle/working/alt/corpus/build_ucf_hmdb_index.py
Output file downloaded to /kaggle/working/alt/corpus/corpus_embeddings.pt
Output file downloaded to /kaggle/working/alt/corpus/corpus_entities.json
Output file downloaded to /kaggle/working/alt/corpus/corpus_meta.json
Output file downloaded to /k

In [16]:
import os
import shutil
import subprocess
import torch

hmdb_splits = [
    "/kaggle/working/alt/corpus/hmdb_val_split1.json",
    "/kaggle/working/alt/corpus/hmdb_val_split2.json",
    "/kaggle/working/alt/corpus/hmdb_val_split3.json",
]

if os.path.exists("/kaggle/working/alt/corpus/hmdb_train_split1.json"):
    print("Note: hmdb_train_split1.json exists but will NOT be used for eval.")

for split_path in hmdb_splits:
    if not os.path.exists(split_path):
        print(f"Missing HMDB split: {split_path}")
        continue
    subprocess.run([
        "python", "/kaggle/working/alt/eval.py",
        "--config", "/kaggle/working/alt/configs/alt_b16.yaml",
        "--checkpoint", "/kaggle/working/runs/alt_b16/best.pt",
        "--split_json", split_path,
        "--label_offset", "101",
        "--n_classes", "51",
] , check=True)

torch.cuda.empty_cache()
print("Cleared CUDA cache after evaluation")

Note: hmdb_train_split1.json exists but will NOT be used for eval.
Loaded checkpoint (epoch 10, best_acc=0.8818)
hmdb_val_split1.json: 0.2327 (356/1530)

Zero-shot accuracy on split: 0.2327
Loaded checkpoint (epoch 10, best_acc=0.8818)
hmdb_val_split2.json: 0.2275 (348/1530)

Zero-shot accuracy on split: 0.2275
Loaded checkpoint (epoch 10, best_acc=0.8818)
hmdb_val_split3.json: 0.2529 (387/1530)

Zero-shot accuracy on split: 0.2529
Cleared CUDA cache after evaluation
